In [1]:
from sklearn.datasets import fetch_california_housing, fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score
import pandas as pd
import numpy as np
from catboost import CatBoostRegressor


def load_and_split(X, y, random_state=42):
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=random_state
    )
    print(X_train.shape, X_test.shape)
    X_train, X_val, y_train, y_val = train_test_split(
        X_train, y_train, test_size=0.2, random_state=random_state
    )
    print(X_train.shape, X_test.shape, X_val.shape)
    X_hp, X_val, y_hp, y_val = train_test_split(
        X_val, y_val, test_size=0.5, random_state=random_state
    )
    print(X_train.shape, X_test.shape, X_val.shape, X_hp.shape)
    return X_train, X_hp, X_val, X_test, y_train, y_hp, y_val, y_test


def fetch_openml_numeric(name, target):
    ds = fetch_openml(name, as_frame=True)
    X = ds.data
    y = ds.target.astype(float)

    # оставляем только числовые признаки
    X = X.select_dtypes(include=[np.number])

    return X.values, y.values


def fetch_openml_numeric_by_id(data_id):
    ds = fetch_openml(data_id=data_id, as_frame=True)
    X = ds.data.select_dtypes(include=[np.number])
    y = ds.target.astype(float)
    return X.values, y.values

In [ ]:
from catboost import CatBoostRegressor
import csv
import os

import sys

sys.path.append("..")

from src.model_generator import CopyModelGenerator, OptunaModelGenerator
from src.gradient_boosting_regressor import MyCatBoost
import optuna

RESULTS_PATH = "benchmark_results_adaptive_lr.csv"


def log_result_csv(model_name, name, r2_test, n_trees, path=RESULTS_PATH):
    file_exists = os.path.isfile(path)

    with open(path, mode="a", newline="") as f:
        writer = csv.writer(f)
        if not file_exists:
            writer.writerow(["model", "dataset", "r2_test", "n_trees"])
        writer.writerow([model_name, name, f"{r2_test:.6f}", n_trees])


def test_treewise_hpo_boosting(name, value):
    X_train, X_hp, X_val, X_test, y_train, y_hp, y_val, y_test = value

    study = optuna.create_study(
        direction="minimize",  # RMSE
        sampler=optuna.samplers.TPESampler(),
    )


    single_tree_model = CatBoostRegressor(
        iterations=1, learning_rate=1.0, loss_function="RMSE", verbose=False
    )
    model_generator = OptunaModelGenerator(single_tree_model, study, n_trials_per_iter=50)
    gbrt = MyCatBoost(
        model_generator=model_generator, n_estimators=2000, learning_rate=0.1, verbose=1
    )

    gbrt.fit(
        X_train,
        y_train,
        hp_set=(X_hp, y_hp),
        eval_set=(X_val, y_val),
        early_stopping_rounds=5,
    )

    pred = gbrt.predict(X_test)
    r2 = r2_score(y_test, pred)
    log_result_csv("treewise HPO (proposal)", name, r2, len(gbrt.models))


def test_catboost(name, value):
    X_train, X_hp, X_val, X_test, y_train, y_hp, y_val, y_test = value

    gbrt = CatBoostRegressor(
        n_estimators=2000, learning_rate=0.1, loss_function="RMSE", verbose=False
    )

    gbrt.fit(
        X_train,
        y_train,
        eval_set=(X_val, y_val),
        early_stopping_rounds=50,
    )

    pred = gbrt.predict(X_test)
    r2 = r2_score(y_test, pred)
    log_result_csv(
        "catboost (default HP)", name, r2, gbrt.get_best_iteration() + 1
    )

def test_catboost_hpo(name, value, n_trials=50):
    X_train, X_hp, X_val, X_test, y_train, y_hp, y_val, y_test = value

    def _sample_params(trial):
        grow_policy = trial.suggest_categorical(
            "grow_policy", ["SymmetricTree", "Depthwise", "Lossguide"]
        )

        params = {
            "iterations": 1,
            "learning_rate": 1.0,
            "loss_function": "RMSE",
            "grow_policy": grow_policy,

            "border_count": trial.suggest_int("border_count", 32, 255),
            "feature_border_type": trial.suggest_categorical(
                "feature_border_type",
                ["GreedyLogSum", "Median", "Uniform"]
            ),
            "l2_leaf_reg": trial.suggest_float(
                "l2_leaf_reg", 1e-6, 100.0, log=True
            ),
            "min_data_in_leaf": trial.suggest_int(
                "min_data_in_leaf", 1, 64
            ),
            "random_strength": trial.suggest_float(
                "random_strength", 1e-3, 10.0, log=True
            ),
            "rsm": trial.suggest_float("rsm", 0.3, 1.0),
            "score_function": trial.suggest_categorical(
                "score_function", ["Cosine", "L2"]
            ),
            "verbose": False,
        }

        if grow_policy in ["SymmetricTree", "Depthwise"]:
            params["depth"] = trial.suggest_int("depth", 2, 12)
        else:
            params["max_leaves"] = trial.suggest_int("max_leaves", 8, 64)

        bootstrap_type = trial.suggest_categorical(
            "bootstrap_type", ["Bayesian", "Bernoulli", "No"]
        )
        params["bootstrap_type"] = bootstrap_type

        if bootstrap_type == "Bernoulli":
            params["subsample"] = trial.suggest_float("subsample", 0.5, 1.0)
        elif bootstrap_type == "Bayesian":
            params["bagging_temperature"] = trial.suggest_float(
                "bagging_temperature", 0.0, 10.0
            )

        return params

    def objective(trial):
        params = _sample_params(trial)
        
        params["iterations"] = 2000
        params["learning_rate"] = 0.1
        params["loss_function"] = "RMSE"
        params["verbose"] = False

        model = CatBoostRegressor(**params)
        model.fit(
            X_train,
            y_train,
            eval_set=(X_val, y_val),
            early_stopping_rounds=50,
        )
        
        pred_hp = model.predict(X_hp)
        rmse_hp = ((y_hp - pred_hp) ** 2).mean() ** 0.5
        return rmse_hp

    study = optuna.create_study(direction="minimize")
    study.optimize(objective, n_trials=n_trials)

    best_params = study.best_trial.params
    best_params["iterations"] = 2000
    best_params["learning_rate"] = 0.1
    best_params["loss_function"] = "RMSE"
    best_params["verbose"] = False

    best_model = CatBoostRegressor(**best_params)
    best_model.fit(
        X_train,
        y_train,
        eval_set=(X_val, y_val),
        early_stopping_rounds=50,
    )

    pred_test = best_model.predict(X_test)
    r2 = r2_score(y_test, pred_test)

    log_result_csv(
        "catboost (HPO)",
        name,
        r2,
        best_model.get_best_iteration() + 1
    )

    return best_model, study


def test_all_3(name, value):
    test_treewise_hpo_boosting(name, value)
    test_catboost_hpo(name, value)
    test_catboost(name, value)

In [ ]:
# 1. California Housing (~20k)
# cal = fetch_california_housing(as_frame=True)
# test_all_3(
#     "california_housing",
#     load_and_split(cal.data.values, cal.target.values),
# )


# # 2. Bike Sharing Demand (~17k)
# X, y = fetch_openml_numeric("Bike_Sharing_Demand", target="count")
# test_all_3(
#     "bike_sharing",
#     load_and_split(X, y),
# )


# # 3. Medical Charges (~13k)
# X, y = fetch_openml_numeric("medical_charges", target="charges")
# test_all_3(
#     "medical_charges",
#     load_and_split(X, y),
# )


# # King County House Prices
# X, y = fetch_openml_numeric_by_id(42165)
# test_all_3(
#     "king_county_house_prices",
#     load_and_split(X, y),
# )


# 1. Online News Popularity (shares)
# ~39k samples, noisy, heavy-tailed target
X, y = fetch_openml_numeric_by_id(42705)
test_all_3(
    "online_news_popularity",
    load_and_split(X, y),
)


# 2. YearPredictionMSD
# ~515k samples, но low-dim, можно сабсемплить
X, y = fetch_openml_numeric_by_id(44027)
test_all_3(
    "year_prediction_msd",
    load_and_split(
        X[:20_000],
        y[:20_000],  # безопасный сабсет
    ),
)


# 3. CPU Activity
# ~20k samples, классический UCI-style regression
X, y = fetch_openml_numeric_by_id(44963)
test_all_3(
    "cpu_activity",
    load_and_split(X, y),
)


# 4. Facebook Comment Volume
# ~50k samples
X, y = fetch_openml_numeric_by_id(4549)
test_all_3(
    "facebook_comment_volume",
    load_and_split(X, y),
)


# 5. Airline Delay (departure delay)
# ~54k samples после очистки
X, y = fetch_openml_numeric_by_id(1169)
mask = np.isfinite(y)
test_all_3(
    "airline_delay",
    load_and_split(X[mask], y[mask]),
)


# 6. Superconductivity
# ~21k samples, физика, сложные взаимодействия
X, y = fetch_openml_numeric_by_id(44964)
test_all_3(
    "superconductivity",
    load_and_split(X, y),
)


# 7. Diamonds (price)
# ~54k samples
X, y = fetch_openml_numeric_by_id(42225)
test_all_3(
    "diamonds",
    load_and_split(X, y),
)


# 8. House Prices (Ames, extended)
# ~29k samples
X, y = fetch_openml_numeric_by_id(42563)
test_all_3(
    "ames_housing_large",
    load_and_split(X, y),
)


# 9. Brazilian Houses
# ~10k samples
X, y = fetch_openml_numeric_by_id(45020)
test_all_3(
    "brazilian_houses",
    load_and_split(X, y),
)


# 10. Metro Interstate Traffic Volume
# ~48k samples, сильная сезонность
X, y = fetch_openml_numeric_by_id(42477)
test_all_3(
    "metro_traffic_volume",
    load_and_split(X, y),
)

[I 2026-01-20 18:59:05,491] A new study created in memory with name: no-name-49ef8c2c-923e-4e40-a9f1-670e676f3208


(320000, 100) (80000, 100)
(256000, 100) (80000, 100) (64000, 100)
(256000, 100) (80000, 100) (32000, 100) (32000, 100)
[0] train RMSE=10.640300, val RMSE=10.770374
[1] train RMSE=10.337470, val RMSE=10.558438
[2] train RMSE=10.055686, val RMSE=10.375614
[3] train RMSE=9.845241, val RMSE=10.222491
[4] train RMSE=9.604373, val RMSE=10.084897
[5] train RMSE=9.405679, val RMSE=9.964458
[6] train RMSE=9.221681, val RMSE=9.864095
[7] train RMSE=9.067436, val RMSE=9.782392
[8] train RMSE=8.949603, val RMSE=9.710127
[9] train RMSE=8.791661, val RMSE=9.644738
[10] train RMSE=8.699345, val RMSE=9.595432
[11] train RMSE=8.593459, val RMSE=9.544638
[12] train RMSE=8.483144, val RMSE=9.498813
[13] train RMSE=8.375243, val RMSE=9.460413
[14] train RMSE=8.273625, val RMSE=9.423446
[15] train RMSE=8.193501, val RMSE=9.392415
[16] train RMSE=8.105415, val RMSE=9.361649
[17] train RMSE=8.035956, val RMSE=9.340101
[18] train RMSE=7.975783, val RMSE=9.318929
[19] train RMSE=7.905626, val RMSE=9.297292
[2

In [ ]:
print(1/0)

In [ ]:
# 4. Facebook Comment Volume
# ~50k samples
X, y = fetch_openml_numeric_by_id(4549)
print(X.shape)
X = X[:300000, :]
y = y[:300000]



name = "facebook_comment_volume"
X_train, X_val, X_hp, X_test, y_train, y_val, y_hp, y_test = load_and_split_2(X, y)
del X, y

(583250, 77)
(240000, 77) (60000, 77)
(192000, 77) (60000, 77) (48000, 77)
(192000, 77) (60000, 77) (24000, 77) (24000, 77)


In [ ]:
from catboost import CatBoostRegressor

model = CatBoostRegressor(iterations=1000, learning_rate=0.1, verbose=1)
model.fit(
    X_train, y_train, eval_set=(X_val, y_val), early_stopping_rounds=50
)

pred = model.predict(X_test)
r2_score(y_test, pred)


0:	learn: 612.7310092	test: 513.6789100	best: 513.6789100 (0)	total: 16.4ms	remaining: 16.4s
1:	learn: 565.3060887	test: 473.4728181	best: 473.4728181 (1)	total: 34.1ms	remaining: 17s
2:	learn: 523.6653712	test: 436.8790549	best: 436.8790549 (2)	total: 51.5ms	remaining: 17.1s
3:	learn: 487.2168340	test: 404.1079353	best: 404.1079353 (3)	total: 68.7ms	remaining: 17.1s
4:	learn: 453.8175639	test: 374.1238333	best: 374.1238333 (4)	total: 87.3ms	remaining: 17.4s
5:	learn: 424.2307561	test: 349.1818797	best: 349.1818797 (5)	total: 103ms	remaining: 17.1s
6:	learn: 398.3332957	test: 328.4185540	best: 328.4185540 (6)	total: 120ms	remaining: 17s
7:	learn: 375.3946487	test: 308.4962455	best: 308.4962455 (7)	total: 136ms	remaining: 16.9s
8:	learn: 354.8594518	test: 290.1908784	best: 290.1908784 (8)	total: 151ms	remaining: 16.6s
9:	learn: 337.0987024	test: 274.9781236	best: 274.9781236 (9)	total: 166ms	remaining: 16.5s
10:	learn: 321.6193495	test: 261.4690059	best: 261.4690059 (10)	total: 182ms	re

0.7759632599754583

In [ ]:
from catboost import CatBoostRegressor
import csv
import os

import sys

sys.path.append("..")

from src.model_generator import CopyModelGenerator, OptunaModelGenerator
from src.gradient_boosting_regressor import MyCatBoost
import optuna

study = optuna.create_study(
    direction="minimize",  # RMSE
    sampler=optuna.samplers.TPESampler(),
)


single_tree_model = CatBoostRegressor(
    iterations=1, learning_rate=1.0, loss_function="RMSE", verbose=False
)
model_generator = OptunaModelGenerator(single_tree_model, study, n_trials_per_iter=50)
gbrt = MyCatBoost(
    model_generator=model_generator, n_estimators=2000, learning_rate=0.1, verbose=1
)

gbrt.fit(
    X_train,
    y_train,
    hp_set=(X_hp, y_hp),
    eval_set=(X_val, y_val),
    early_stopping_rounds=5,
)

pred = gbrt.predict(X_test)
r2 = r2_score(y_test, pred)
r2_val = r2_score(y_val, gbrt.predict(X_val))

[I 2026-01-20 14:10:51,569] A new study created in memory with name: no-name-0f59e46a-d4f8-432f-bc60-277ff108e55e


[0] train RMSE=603.073543, val RMSE=508.194763
[1] train RMSE=549.048819, val RMSE=462.996057
[2] train RMSE=498.698438, val RMSE=420.711287
[3] train RMSE=454.433987, val RMSE=385.273067
[4] train RMSE=413.866002, val RMSE=352.668644
[5] train RMSE=377.460406, val RMSE=324.660515
[6] train RMSE=348.578299, val RMSE=299.114272
[7] train RMSE=320.828450, val RMSE=276.673466
[8] train RMSE=299.530525, val RMSE=257.847830
[9] train RMSE=278.495034, val RMSE=241.300724
[10] train RMSE=259.713879, val RMSE=226.274998
[11] train RMSE=244.265094, val RMSE=214.170259
[12] train RMSE=228.638232, val RMSE=204.175105
[13] train RMSE=215.085002, val RMSE=193.698786
[14] train RMSE=204.454770, val RMSE=185.998479
[15] train RMSE=192.944509, val RMSE=178.054081
[16] train RMSE=184.641632, val RMSE=171.767695
[17] train RMSE=178.055970, val RMSE=166.998063
[18] train RMSE=171.392584, val RMSE=162.086451
[19] train RMSE=164.096512, val RMSE=158.003245
[20] train RMSE=157.076123, val RMSE=153.363925
[2

KeyboardInterrupt: 

: 

In [ ]:
print(r2)

0.09209261782261502
